# TF-IDF + Transformer ensemble (Colab)

Runs the TF-IDF baseline and ensembles it with the transformer by averaging
per-class probabilities. TF-IDF trains in seconds, so we train it fresh here
(no `.joblib` to ship -> no sklearn pickle-version issues).

**Probability contract** — every model outputs a CSV with columns:
`id, proba_Cardiology, proba_Neurology, proba_Orthopedics, proba_Gastroenterology, proba_Other`
where `id` matches `data/splits/val.csv` (val) or the test file (submission).

## 1. Get the code + data (already on the `jin-dev` branch)
No `pip install -r requirements.txt` needed — Colab already has scikit-learn /
pandas / numpy, and that's all the TF-IDF code uses.

In [ ]:
!git clone -b jin-dev https://github.com/OmMane1/codametrix-clinical-classifier.git
%cd codametrix-clinical-classifier

## 2. TF-IDF side — train on the shared split, output val probabilities

In [ ]:
!python -m clinical_classifier.train --data data/splits/train.csv --out models/tfidf_baseline.joblib
!python -m clinical_classifier.predict --model models/tfidf_baseline.joblib \
    --data data/splits/val.csv --proba --out tfidf_val_proba.csv
import pandas as pd; pd.read_csv('tfidf_val_proba.csv').head()

## 3. Transformer side — produce val probabilities in the SAME format

Run your transformer on `data/splits/val.csv` and save `transformer_val_proba.csv`
with the same `id` + `proba_<Class>` columns (exact class names).

In [ ]:
# >>> your existing transformer inference goes here <<<
# write transformer_val_proba.csv: id, proba_Cardiology, proba_Neurology,
#   proba_Orthopedics, proba_Gastroenterology, proba_Other  (id matches val.csv)
transformer_val_proba = 'transformer_val_proba.csv'

## 4. Tune the ensemble weight on the frozen val set

In [ ]:
!python ensemble.py --proba tfidf_val_proba.csv transformer_val_proba.csv \
    --truth data/splits/val.csv --sweep

## 5. Final submission on the hidden test

When the hidden test arrives (`test.csv` with `id,text`): retrain TF-IDF on ALL
data, predict, and average with the transformer using the best weight from step 4.

In [ ]:
!python -m clinical_classifier.train --data data/mt_specialty.csv --out models/tfidf_submission.joblib
!python -m clinical_classifier.predict --model models/tfidf_submission.joblib \
    --data test.csv --proba --out tfidf_test_proba.csv
# transformer predicts test.csv -> transformer_test_proba.csv (same format), then:
!python ensemble.py --proba tfidf_test_proba.csv transformer_test_proba.csv \
    --weights 0.4 0.6 --out submission.csv   # <- weight from step 4
import pandas as pd; pd.read_csv('submission.csv').head()

## 6. (Optional) Power the explainable demo with the ensemble

Download `transformer_val_proba.csv`, drop it next to `app.py` in your local
repo, and rerun `streamlit run app.py` (or `python make_demo.py`). The demo
auto-detects the file and switches to ENSEMBLE mode: decision + confidence from
the blend, word highlights from the interpretable TF-IDF component. Set the
TF-IDF weight (sidebar slider / `W_TFIDF`) to the best value from step 4.

In [ ]:
from google.colab import files
files.download('transformer_val_proba.csv')